# Wall-to-wall CBI
This code will run the Parks et al. script for bcCBI calculations to create a wall-to-wall dataset that can later be masked to any set of fire perimeter polygons. By default this script will run for the western 17 states of CONUS. Note that Sean Parks' CBI model is only designed to be used for calculating CBI in FORESTED regions, and thus subsequent fire perimeters should be filtered accordingly. The wall-to-wall dataset will be created for every year from 1985 through to the most recent year with summer imagery (based on the day of run being in November or later). It will skip any years for which the asset already exists in the project directory.

Tyler L. McIntosh, CU Boulder, 2025

Code adapted from:
Parks SA, Holsinger LM, Koontz MJ, Collins, Whitman E, et al. 2019. Giving ecological meaning to satellite-derived fire severity metrics across North American forests within Google Earth Engine. Remote Sensing 2019, 11.

Original code link: https://code.earthengine.google.com/b4fb68fb7f8f883595dbe165ff82e0d9

Updated (2022) code link from Sean Parks: https://code.earthengine.google.com/484e28e8707a2b6031bea2de35272daa


In [36]:
import os
import re
from datetime import datetime

import ee
import geemap



#####################################
#    SET ALL PARAMETERS #############
#####################################

PROJECT_DIR = "projects/ee-tymc5571-goodfire/assets/"
GEE_PROJECT = "ee-tymc5571-goodfire"
ASSET_STRING = "west_wall_to_wall_bccbi"
CRS = "EPSG:4326" #this is the CRS that the data will be exported in

ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

# Set output region as desired
states = ee.FeatureCollection("TIGER/2018/States")

western_states = [
    'WA', 'OR', 'CA', 'ID', 'NV', 'MT', 'WY', 'UT',
    'CO', 'AZ', 'NM', 'ND', 'SD', 'NE', 'KS', 'OK', 'TX'
]
outputRegion = states.filter(
    ee.Filter.inList('STUSPS', western_states)
)

# # Code for CONUS if desired
# excluded = ['AK', 'HI', 'PR', 'GU', 'VI', 'MP', 'AS', 'UM']
# outputRegion = states.filter(
#     ee.Filter.inList('STUSPS', excluded).Not()
# )


# //----------        IMAGERY SELECTION        ---------------------------//
# // Select the imagery to export from the suite of available indices below.
# // Add the VARIABLE NAMES (as desired) to brackets below, using quotes, e.g.  ['CBI', 'CBI_bc', 'dnbr', 'rbr']

# //    VARIABLE NAME     DESCRIPTION
# //    CBI               Composite Burn Index
# //    CBI_bc            Bias-corrected Composite Burn Index
# //    dnbr              delta normalized burn ratio
# //    rbr               relativized burn ratio
# //    rdnbr             relativized delta normalized burn ratio
# ///   dndvi             delta normalized differenced vegetation index
# //    devi              delta enhanced vegetation index
# //    dndmi             delta normalized difference moisture index
# //    dmirbi            delta mid-infrared bi-spectral index
# //    post_nbr          post-fire normalized burn ratio
# //    post_mirbi        post-fire mid-infrared bi-spectral index
bandsToExport = ['CBI_bc'];




In [37]:
# Visualize output region to confirm it is correct
Map = geemap.Map(center=[37.5, -96], zoom=4)
Map.addLayer(outputRegion, {'color': 'blue'}, 'Selected output region')
Map

Map(center=[37.5, -96], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(c…

In [38]:
# FUNCTIONS

def list_assets(parent: str) -> list[str]:
    """
    Recursively list asset IDs under an Earth Engine folder/collection.

    Args:
        parent: e.g. "projects/ee-tymc5571-goodfire/assets/"
    Returns:
        A list of child asset IDs under the given parent.
    Requires:
        import ee
        from typing import List
    """
    assets = []
    def _list(parent_id):
        response = ee.data.listAssets({'parent': parent_id})
        for item in response.get('assets', []):
            asset_id = item.get('name')
            asset_type = item.get('type')
            if asset_type in ('FOLDER', 'IMAGE_COLLECTION'):
                _list(asset_id)
            else:
                assets.append(asset_id)

    _list(parent)
    return assets

def extract_years_from_assets(asset_list):
    """
    Extracts 4-digit years from the END of asset names.
    E.g., '.../welty_wildfire_fri_splitfrg5_1989' -> 1989
    Ignores names with year ranges or non-matching patterns.
    """
    years = []
    pattern = re.compile(r'_(\d{4})$')  # Match ending with '_YYYY'
    for asset in asset_list:
        match = pattern.search(asset)
        if match:
            years.append(int(match.group(1)))
    return ee.List(sorted(set(years)))

In [39]:
start_year = 1985
now = datetime.now()
end_year = now.year if now.month >= 11 else now.year - 1


# Generate landsat years
viable_years = ee.List.sequence(start_year, end_year).map(
    lambda y: ee.Number(y).format().slice(0, -2)
)

print('Viable Landsat years:', viable_years.getInfo())

assets = list_assets(PROJECT_DIR)
filtered_assets = [a for a in assets if ASSET_STRING in a]
print(assets)

available_years = extract_years_from_assets(filtered_assets).map(
    lambda y: ee.Number(y).format()
)

# Years with assets
years_with_assets = viable_years.map(
    lambda y: ee.Algorithms.If(available_years.contains(y), y, None)
).removeAll([None])

# Years missing assets
years_missing_assets = viable_years.map(
    lambda y: ee.Algorithms.If(ee.Algorithms.IsEqual(available_years.contains(y), False), y, None)
).removeAll([None])

print("Landsat years with pre-existing assets:", years_with_assets.getInfo())
print("Landsat years missing assets:", years_missing_assets.getInfo())

uniqueYears = years_missing_assets

Viable Landsat years: ['1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']
['projects/ee-tymc5571-goodfire/assets/WUMI_20250509_main_fires_unified_simple', 'projects/ee-tymc5571-goodfire/assets/West_NFPORS_2010_2021', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_1986', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_1987', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_1988', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_1989', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_1990', 'projects/ee-tymc5571-goodfire/assets/all_gf_welty_wildfire_fri_splitfrg5_1991', 'projects/ee-tymc5571-goodfi

In [40]:
# TESTING BLOCK

#################
# counties = ee.FeatureCollection("TIGER/2018/Counties")
# outputRegion = counties.filter(
#     ee.Filter.And(
#         ee.Filter.eq('STATEFP', '08'),  # Colorado
#         ee.Filter.eq('NAME', 'Boulder')
#     )
# )

uniqueYears = ee.List([ee.Number(2003).format()])
print(uniqueYears.getInfo())

#################

['2003']


In [41]:
# CODE FROM PARKS ET AL

################################################################################
#----------------  RANDOM FOREST SPECIFICATIONS---------------------#
# Bands for the random forest classification
rf_bands = ['def', 'lat',  'rbr', 'dmirbi', 'dndvi', 'post_mirbi']

# Load training data for Random Forest classification
cbi = ee.FeatureCollection("users/grumpyunclesean/CBI_predictions/data_for_ee_model")

# Load climatic water deficit variable (def) for random forest classification
def_img = ee.Image("users/grumpyunclesean/CBI_predictions/def").rename('def').toInt()

# Create latitude image for random forest classification
lat = ee.Image.pixelLonLat().select('latitude').rename('lat').round().toInt()

# Parameters for random forest classification
nrow_training_fold = cbi.size()        # number of training observations
minLeafPopulation  = nrow_training_fold.divide(75).divide(6).round()

# Random forest classifier
fsev_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=500,
    minLeafPopulation=minLeafPopulation,
    seed=123
).train(cbi, 'CBI', rf_bands).setOutputMode('REGRESSION')

################################################################################
#--------------------     PROCESSING     ----------------------------#
#  Suite of spectral indices available for export.
bandList = ['dnbr', 'rbr', 'rdnbr', 'dndvi', 'devi', 'dndmi', 'dmirbi', 'post_nbr', 'post_mirbi', 'CBI', 'CBI_bc']

################################
#  GET LANDSAT COLLECTIONS
################################
ls9SR = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
ls8SR = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
ls7SR = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')
ls5SR = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
ls4SR = ee.ImageCollection('LANDSAT/LT04/C02/T1_L2')

print('LS Projection', ls9SR.first().projection().getInfo())

#########################################
# FUNCTIONS TO CREATE SPECTRAL INDICES
#########################################
# Apply scaling factors.
def applyScaleFactors(lsImage):
    opticalBands = lsImage.select(["SR_B."]).multiply(0.0000275).add(-0.2)
    return lsImage.addBands(opticalBands, None, True)

# Returns vegetation indices for LS8 and LS9
def ls8_9_Indices(lsImage):
    nbr  = lsImage.normalizedDifference(['SR_B5', 'SR_B7']).toFloat()
    ndvi = lsImage.normalizedDifference(['SR_B5', 'SR_B4']).toFloat()
    ndmi = lsImage.normalizedDifference(['SR_B5', 'SR_B6']).toFloat()
    evi  = lsImage.expression(
        '2.5 * ((SR_B5 - SR_B4) / (SR_B5 + 6 * SR_B4 - 7.5 * SR_B2 + 1))',
        {'SR_B5': lsImage.select('SR_B5'),
         'SR_B4': lsImage.select('SR_B4'),
         'SR_B2': lsImage.select('SR_B2')}
    ).toFloat()
    mirbi = lsImage.expression(
        '((10 * SR_B6) - (9.8 * SR_B7) + 2)',
        {'SR_B6': lsImage.select('SR_B6'),
         'SR_B7': lsImage.select('SR_B7')}
    ).toFloat()
    qa = lsImage.select(['QA_PIXEL'])
    return (nbr.addBands([ndvi, ndmi, evi, mirbi, qa])
               .select([0,1,2,3,4,5], ['nbr','ndvi','ndmi','evi','mirbi','QA_PIXEL'])
               .copyProperties(lsImage, ['system:time_start']))

# Returns indices for LS4, LS5 and LS7
def ls4_7_Indices(lsImage):
    nbr  = lsImage.normalizedDifference(['SR_B4', 'SR_B7']).toFloat()
    ndvi = lsImage.normalizedDifference(['SR_B4', 'SR_B3']).toFloat()
    ndmi = lsImage.normalizedDifference(['SR_B4', 'SR_B5']).toFloat()
    evi = lsImage.expression(
        '2.5 * ((SR_B4 - SR_B3) / (SR_B4 + 6 * SR_B3 - 7.5 * SR_B1 + 1))',
        {'SR_B4': lsImage.select('SR_B4'),
         'SR_B3': lsImage.select('SR_B3'),
         'SR_B1': lsImage.select('SR_B1')}
    ).toFloat()
    mirbi = lsImage.expression(
        '((10 * SR_B5) - (9.8 * SR_B7) + 2)',
        {'SR_B5': lsImage.select('SR_B5'),
         'SR_B7': lsImage.select('SR_B7')}
    ).toFloat()
    qa = lsImage.select(['QA_PIXEL'])
    return (nbr.addBands([ndvi, ndmi, evi, mirbi, qa])
               .select([0,1,2,3,4,5], ['nbr','ndvi','ndmi','evi','mirbi','QA_PIXEL'])
               .copyProperties(lsImage, ['system:time_start']))

###############################################
# FUNCTION TO MASK CLOUD, WATER, SNOW, ETC.
###############################################
def lsCfmask(lsImage):
    # Bits 3,4,5,7: cloud, cloud-shadow, snow, water respectively.
    cloudsBitMask      = (1 << 3)
    cloudShadowBitMask = (1 << 4)
    snowBitMask        = (1 << 5)
    waterBitMask       = (1 << 7)
    # Get the pixel QA band.
    qa = lsImage.select('QA_PIXEL')
    # Flags should be set to zero, indicating clear conditions.
    clear = (qa.bitwiseAnd(cloudsBitMask).eq(0)
               .And(qa.bitwiseAnd(cloudShadowBitMask).eq(0))
               .And(qa.bitwiseAnd(snowBitMask).eq(0))
               .And(qa.bitwiseAnd(waterBitMask).eq(0)))
    return (lsImage.updateMask(clear)
            .select([0,1,2,3,4])
            .copyProperties(lsImage, ["system:time_start"]))

# Create water mask from Hansen's Global Forest Change to use in processing function
waterMask = ee.Image('UMD/hansen/global_forest_change_2015').select(['datamask']).eq(1)

###############################################
# RUN FUNCTIONS ON LANDSAT COLLECTION
###############################################
ls9 = ls9SR.map(applyScaleFactors).map(ls8_9_Indices).map(lsCfmask)
ls8 = ls8SR.map(applyScaleFactors).map(ls8_9_Indices).map(lsCfmask)
ls7 = ls7SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)
ls5 = ls5SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)
ls4 = ls4SR.map(applyScaleFactors).map(ls4_7_Indices).map(lsCfmask)

# Merge Landsat Collections
lsCol = ee.ImageCollection(ls9.merge(ls8).merge(ls7).merge(ls5).merge(ls4))

# ------------------ Create Spectral Imagery for a region -----------------#
# regionF is a FeatureCollection or Feature, fireYear/startday/endday are strings (per original)

def specImagery(regionF, fireYear, startday, endday):
    # create pre- and post-fire imagery
    fireYear = ee.Date.parse('YYYY', fireYear)
    startday = ee.Number.parse(startday)
    endday   = ee.Number.parse(endday)
    region = regionF.geometry().bounds()

    # Pre-Imagery
    preFireYear = fireYear.advance(-1, 'year')
    preFireIndices = ee.Algorithms.If(
        lsCol.filterBounds(region)
             .filterDate(preFireYear, fireYear)
             .filter(ee.Filter.dayOfYear(startday, endday))
             .size(),
        lsCol.filterBounds(region)
             .filterDate(preFireYear, fireYear)
             .filter(ee.Filter.dayOfYear(startday, endday))
             .mean()
             .select([0,1,2,3,4], ['pre_nbr','pre_ndvi','pre_ndmi','pre_evi','pre_mirbi']),
        ee.Image.cat(ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image())
               .rename(['pre_nbr','pre_ndvi','pre_ndmi','pre_evi','pre_mirbi'])
    )

    # if any pixels within fire have only one 'scene' or less, add additional year backward
    preFireYear2 = fireYear.advance(-2, 'year')
    preFireIndices2 = ee.Algorithms.If(
        lsCol.filterBounds(region)
             .filterDate(preFireYear2, fireYear)
             .filter(ee.Filter.dayOfYear(startday, endday))
             .size(),
        lsCol.filterBounds(region)
             .filterDate(preFireYear2, fireYear)
             .filter(ee.Filter.dayOfYear(startday, endday))
             .mean()
             .select([0,1,2,3,4], ['pre_nbr','pre_ndvi','pre_ndmi','pre_evi','pre_mirbi']),
        ee.Image.cat(ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image())
               .rename(['pre_nbr','pre_ndvi','pre_ndmi','pre_evi','pre_mirbi'])
    )
    pre_filled = ee.Image(preFireIndices).unmask(preFireIndices2)

    # Post-Imagery
    postFireYear = fireYear.advance(1, 'year')
    postFireIndices = ee.Algorithms.If(
        lsCol.filterBounds(region)
             .filterDate(postFireYear, fireYear.advance(2, 'year'))
             .filter(ee.Filter.dayOfYear(startday, endday))
             .size(),
        lsCol.filterBounds(region)
             .filterDate(postFireYear, fireYear.advance(2, 'year'))
             .filter(ee.Filter.dayOfYear(startday, endday))
             .mean()
             .select([0,1,2,3,4], ['post_nbr','post_ndvi','post_ndmi','post_evi','post_mirbi']),
        ee.Image.cat(ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image())
               .rename(['post_nbr','post_ndvi','post_ndmi','post_evi','post_mirbi'])
    )
    # if any pixels within fire have only one 'scene' or less, add additional year forward
    postFireIndices2 = ee.Algorithms.If(
        lsCol.filterBounds(region)
             .filterDate(postFireYear, fireYear.advance(3, 'year'))
             .filter(ee.Filter.dayOfYear(startday, endday))
             .size(),
        lsCol.filterBounds(region)
             .filterDate(postFireYear, fireYear.advance(3, 'year'))
             .filter(ee.Filter.dayOfYear(startday, endday))
             .mean()
             .select([0,1,2,3,4], ['post_nbr','post_ndvi','post_ndmi','post_evi','post_mirbi']),
        ee.Image.cat(ee.Image(), ee.Image(), ee.Image(), ee.Image(), ee.Image())
               .rename(['post_nbr','post_ndvi','post_ndmi','post_evi','post_mirbi'])
    )
    post_filled = ee.Image(postFireIndices).unmask(postFireIndices2)
    fireIndices = pre_filled.addBands(post_filled)

    # dNBR
    burnIndices = fireIndices.expression("(b('pre_nbr') - b('post_nbr')) * 1000") \
                              .rename('dnbr').toInt().addBands(fireIndices)

    # RBR
    burnIndices2 = burnIndices.expression("b('dnbr') / (b('pre_nbr') + 1.001)") \
                               .rename('rbr').toInt().addBands(burnIndices)

    # RdNBR
    burnIndices3 = burnIndices2.expression(
        "abs(b('pre_nbr')) < 0.001 ? 0.001 : b('pre_nbr')"
    ).abs().sqrt().rename('pre_nbr2').toFloat().addBands(burnIndices2)

    burnIndices4 = burnIndices3.expression("b('dnbr') / b('pre_nbr2')") \
                                .rename('rdnbr').toInt().addBands(burnIndices3)

    # dNDVI
    burnIndices5 = burnIndices4.expression("(b('pre_ndvi') - b('post_ndvi')) * 1000") \
                                .rename('dndvi').toInt().addBands(burnIndices4)

    # dEVI
    burnIndices6 = burnIndices5.expression("(b('pre_evi') - b('post_evi')) * 1000") \
                                .rename('devi').toInt().addBands(burnIndices5)

    # dNDMI
    burnIndices7 = burnIndices6.expression("(b('pre_ndmi') - b('post_ndmi')) * 1000") \
                                .rename('dndmi').toInt().addBands(burnIndices6)

    # dMIRBI
    burnIndices8 = burnIndices7.expression("(b('pre_mirbi') - b('post_mirbi')) * 1000") \
                                .rename('dmirbi').toInt().addBands(burnIndices7)

    # Multiply post_mirbi by 1000 to match CBI plot extractions scale
    post_mirbi_1000 = burnIndices8.select("post_mirbi").multiply(1000).toInt()
    burnIndices8 = burnIndices8.addBands(post_mirbi_1000, None, True)

    # add climatic water deficit (def)
    burnIndices9 = burnIndices8.addBands(def_img)

    # add latitude
    burnIndices10 = burnIndices9.addBands(lat)

    # Classify with RF; set precision to two decimals
    cbi_rf = (burnIndices10.select(rf_bands)
              .classify(fsev_classifier)
              .rename('CBI').toFloat()
              .multiply(100).floor().divide(100))  # 2-decimal precision

    burnIndices11 = cbi_rf.addBands(burnIndices10)

    # Bias-corrected CBI
    def bias_correct(bandImage):
        cbi_lo = bandImage.expression("((b('CBI') - 1.5) * 1.3)  + 1.5")
        cbi_hi = bandImage.expression("((b('CBI') - 1.5) * 1.175) + 1.5")
        cbi_mg = bandImage.where(bandImage.lte(1.5), cbi_lo).where(bandImage.gt(1.5), cbi_hi)
        return (cbi_mg.where(cbi_mg.lt(0), 0)
                      .where(cbi_mg.gt(3), 3)
                      .multiply(100).floor().divide(100)  # 2-decimal precision
                      .rename('CBI_bc'))

    validMask = (pre_filled.select('pre_nbr')
                 .add(10)
                 .add(post_filled.select('post_nbr'))
                 .add(10))  # avoid zero => masked
    mask = waterMask.updateMask(validMask)
    burnIndices12 = bias_correct(burnIndices11.select('CBI')).addBands(burnIndices11)
    burnIndices12 = (burnIndices12
                     .select(bandList, bandList)
                     .updateMask(mask)
                     .clip(regionF))

    return burnIndices12

LS Projection {'type': 'Projection', 'crs': 'EPSG:32628', 'transform': [30, 0, 340185, 0, -30, 8808615]}


In [42]:
# Wall-to-wall code

# Function to derive CBI over the entire western US for a given year (fireYear string)
def specImageryFullRegion(fireYear):
    studyAreaNormalGrow = states.filter(ee.Filter.inList(
        'NAME', ['Washington','Oregon','California','Idaho','Montana','Nevada','Colorado','Wyoming','Utah', 'North Dakota', 'South Dakota', 'Nebraska', 'Kansas', 'Oklahoma']))
    studyAreaSpecialGrow = states.filter(ee.Filter.inList(
        'NAME', ['New Mexico','Arizona', 'Texas']))  # different growing seasons per Parks et al. 2019, with Texas added due to its proximity
    startDayNormal  = ee.String('152')  # start of growing season (julian day)
    endDayNormal    = ee.String('258')  # end of growing season (julian day)
    startDaySpecial = ee.String('91')
    endDaySpecial   = ee.String('181')

    normalGrow  = specImagery(studyAreaNormalGrow,  fireYear, startDayNormal,  endDayNormal)
    specialGrow = specImagery(studyAreaSpecialGrow, fireYear, startDaySpecial, endDaySpecial)

    combinedYear = ee.ImageCollection([normalGrow, specialGrow]).mosaic()
    combinedYear = combinedYear.set({'Fire_Year': fireYear})
    return combinedYear

years = [str(y) for y in uniqueYears.getInfo()]

for year_str in years:
    # Build the full-region spectral image for this year
    # (specImageryFullRegion expects a year string; keep as string to match parse('YYYY')).
    img = (specImageryFullRegion(year_str)
           .clip(outputRegion)                          # ensure we only keep outputRegion polygons
           .set({'year': ee.Number.parse(year_str)}))   # add 'year' property

    # One asset per year: ASSET_STRING_YYYY
    desc     = f"{ASSET_STRING}_{year_str}"
    asset_id = f"{PROJECT_DIR}{ASSET_STRING}_{year_str}"

    print(asset_id)

    task = ee.batch.Export.image.toAsset(
        image=img.toFloat(),
        description=desc,
        assetId=asset_id,
        scale=30,
        crs=CRS,
        region=outputRegion.geometry(),
        maxPixels=1e13
    )
    task.start()
    print(f"Started export task for {year_str}: {desc}")

projects/ee-tymc5571-goodfire/assets/west_wall_to_wall_bccbi_2003
Started export task for 2003: west_wall_to_wall_bccbi_2003


In [35]:
# vis = {
#     'min': 0,
#     'max': 3,
#     'palette': ['blue', 'yellow', 'red']
# }

# TestMap = geemap.Map(center=[37.5, -96], zoom=4)
# TestMap.addLayer(outputRegion, {'color': 'blue'}, 'Selected output region')
# TestMap.addLayer(img.select('CBI_bc').toFloat(), vis, 'TEST_BCCBI')
# TestMap

In [ ]:
# DEPRECATED CODE FOR USING POLYGONS


# # -------------------------------------------------------------------
# # Use the input polygons as masks on the regional calculations
# # after reducing to image; keep only 'CBI_bc' in the final stack
# data2 = uniqueYears.map(lambda thisYear: (
#     # thisYear is an ee.String due to the earlier .format().slice()
#     (lambda year_str:
#         (lambda _yearFires, _regionData, _fireMask, _maskedData:
#             _maskedData.select('CBI_bc')
#         )(
#             fires.filter(ee.Filter.eq(fireYearAttribute, ee.Number.parse(year_str))),
#             specImageryFullRegion(year_str),
#             fires.filter(ee.Filter.eq(fireYearAttribute, ee.Number.parse(year_str))).reduceToImage(
#                 properties=[fireYearAttribute],
#                 reducer=ee.Reducer.count()
#             ).gt(0),
#             specImageryFullRegion(year_str).updateMask(
#                 fires.filter(ee.Filter.eq(fireYearAttribute, ee.Number.parse(year_str))).reduceToImage(
#                     properties=[fireYearAttribute],
#                     reducer=ee.Reducer.count()
#                 ).gt(0)
#             )
#         )
#     )(thisYear)
# ))

# print(uniqueYears.getInfo())
# def prependYear(year):
#     return ee.String('year_').cat(year)
# uniqueYearNames = uniqueYears.map(prependYear)
# print(uniqueYearNames.getInfo())

# fullColl = ee.ImageCollection(data2).toBands().rename(uniqueYearNames)

# # Export to Asset (single multiband image with one band per year: 'year_XX' per original)
# task = ee.batch.Export.image.toAsset(
#     image=fullColl.toFloat(),  # casting to float maintains NA values in masked pixels
#     description='goodFire_all_cbi_bc' + '_' + str(startYear) + '_' + str(endYear),
#     assetId='goodFire_all_cbi_bc' + '_' + str(startYear) + '_' + str(endYear),
#     scale=30,
#     crs=selectedCRS,
#     region=outputRegion.geometry(),  # geometry of study area
#     maxPixels=1e13
# )
# task.start()